# Milestone 2

In [ ]:
from datasets import load_dataset
# Assume train.csv is in data
dataset = load_dataset('csv', data_files={'train': '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'})
def combine_cols(example):
    example['combined_text'] = str(example['prompt']) + " " + str(example['A'])
    return example
dataset = dataset.map(combine_cols)
print("Length at 51:", len(dataset['train'][51]['combined_text'])) # 614

In [ ]:
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
print("Vocab size:", tokenizer.vocab_size) # 30522
print("SEP ID:", tokenizer.sep_token_id) # 102

In [ ]:
tokens = tokenizer(dataset['train']['prompt'], padding='max_length', truncation=True, max_length=128, return_tensors='pt')
print("Shape:", tokens['input_ids'].shape) # (2000, 128)

In [ ]:
# 12 heads, 768 hidden
print("Head dim:", 768 // 12) # 64

In [ ]:
model = AutoModel.from_pretrained('bert-base-uncased')
inputs = tokenizer(dataset['train'][0]['prompt'], return_tensors='pt')
outputs = model(**inputs)
print("Hidden state shape:", outputs.last_hidden_state.shape) # [1, 31, 768]
print("Sum first 5:", outputs.last_hidden_state[0, 0, :5].sum().item()) # -1.2001

In [ ]:
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
inputs = tokenizer("Light-ion fusion is a technique.", return_tensors='pt')
outputs = model_attn(**inputs)
attn = outputs.attentions[-1]
# Get fusion index
fusion_idx = inputs.input_ids[0].tolist().index(tokenizer.encode("fusion", add_special_tokens=False)[0])
print("Attn weight:", attn[0, 0, 0, fusion_idx].item()) # 0.1025

In [ ]:
from sentence_transformers import SentenceTransformer, util
sbert = SentenceTransformer('all-MiniLM-L6-v2')
p_emb = sbert.encode(dataset['train'][0]['prompt'])
b_emb = sbert.encode(dataset['train'][0]['B'])
print("Cosine sim:", util.cos_sim(p_emb, b_emb).item()) # 0.7658

In [ ]:
# Zero-shot pipeline
from transformers import pipeline
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row1 = dataset['train'][1]
res = zs(row1['prompt'], candidate_labels=[str(row1['A']), str(row1['B']), str(row1['C'])])
print("Top prob:", res['scores'][0]) # 0.4575

res_multi = zs(row1['prompt'], candidate_labels=[str(row1['A']), str(row1['B']), str(row1['C'])], multi_label=True)
diff = abs(sum(res['scores']) - sum(res_multi['scores']))
print("Diff:", diff) # 0.9994

In [ ]:
t2t = pipeline("text2text-generation", model="google/flan-t5-small")
prompt = f"Question: {dataset['train'][0]['prompt']}. Is the correct answer A: {dataset['train'][0]['A']} or B: {dataset['train'][0]['B']}? Answer with just the letter A or B."
out = t2t(prompt, max_new_tokens=5)
print("Output:", out[0]['generated_text']) # B